# Lab 7.5 &mdash; Route: Decide What Happens Next

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Use the real model as a classifier over a CLOSED label set
- Constrain its answer + escalate negative or unknown cases to a human
- Emit a small label that drives the next branch of the workflow

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Route — decide what happens next](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-07-05")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
**Route** decides what happens next and emits a **label from a fixed set** that drives a branch
(deck slide 11): which team, how urgent, auto-handle or escalate. The **closed list** is the trick that
makes an **LLM a reliable classifier**: ask the real model to pick exactly one label, include an **escape
hatch** (`other`), and **constrain** its answer to the set so a stray reply is safe. A deterministic
**fallback path** still routes unhappy or unknown cases to a human. (Routing to a specialist *agent*
graph is Module 8; here the model classifies and rule-based logic escalates.)

In [ ]:
TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}
print("closed label set -> team:", TEAMS)
print("the model picks ONE label; the escape hatch + escalation keep it safe")

## Build it
Complete `classify` (constrain the model's answer to the closed `LABELS`) and `route` (team from the closed
map + when to **escalate**).

In [ ]:
from langchain_core.prompts import PromptTemplate

TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}
LABELS = ("refund", "delivery", "cancel", "other")

CLASSIFY = PromptTemplate.from_template(
    "Classify the customer message into EXACTLY ONE label from: refund, delivery, cancel, other.\n"
    "Use ONLY the message text; do not call any tool or search the web. Reply with only the label word.\n"
    "Message: {msg}\nLabel:")

def classify(message):
    """Use the REAL model as a closed-set classifier."""
    raw = with_backoff(lambda: llm.invoke(CLASSIFY.format(msg=message)).content).strip().lower()
    # constrain the model to the CLOSED set -- a stray answer falls to the escape hatch
    return ___   # TODO: the first LABEL that appears in raw, else "other"

def route(message, sentiment="neutral"):
    intent = classify(message)
    team = ___   # TODO: the team for intent, defaulting to "general" (the escape hatch)
    # escalate to a human when the customer is unhappy OR the intent is unknown
    escalate = ___   # TODO: True if sentiment is negative or intent not in TEAMS
    return {"intent": intent, "team": team, "escalate": escalate}

## Run it for real &amp; read the trace
Route a few real messages: the model classifies each into the closed set, and the escape hatch catches nonsense.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        for msg, sent in [("I want a refund", "neutral"),
                          ("where is my order 4471? it's late", "negative"),
                          ("please cancel order 5090", "neutral"),
                          ("blah blah nonsense", "neutral")]:
            print(f"{msg[:32]:34} ->", route(msg, sent))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The **real model** classifies the message, but `classify` **constrains** its answer to `LABELS` &mdash; the escape hatch (`other`) makes even a stray reply safe.
- A **negative** sentiment escalates to a human even when the team is known &mdash; the deterministic high-stakes fallback path.
- The emitted `{intent, team, escalate}` is the small, closed label the rest of the workflow branches on.

## Your turn (open task &mdash; no grader)
Try messages the model might find ambiguous (*"I'm not sure, maybe cancel, maybe not"*) and see which label it
picks. **What good looks like:** the answer is always in the closed set, ambiguous or unhappy cases escalate, and
you can defend why the escape hatch + escalation matter more than the classifier being perfect.

---
### Key takeaway
A closed label set + an escape hatch is what turns the model into a reliable router; deterministic escalation keeps humans in the high-stakes loop. Routing to specialist AGENTS is the bridge to Module 8.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>